# Precision-focused detector comparison

The original Isolation Forest produced many false alerts on the SKAB validation recordings. This comparison examines whether a more selective detector can reduce those alerts while retaining useful anomaly detection. Precision is the share of alerted readings that are annotated anomalous. Recall is the share of annotated anomaly readings that trigger an alert.

## Why the model changed

The active detector is now gradient boosting, implemented with scikit-learn's `HistGradientBoostingClassifier`. The original Isolation Forest remains available as the baseline. The change also introduces supervised learning: the new model uses normal and anomalous training examples, while Isolation Forest was fitted on normal examples only.

| Aspect | Original Isolation Forest | Selected gradient-boosting model |
|---|---|---|
| Learning approach | Unsupervised detector fitted on normal readings | Supervised classifier fitted on labeled normal and anomalous readings |
| Training input | Eight sensor measurements per reading | 32 relative and rolling features derived from the same eight sensors |
| Training observations | 18,306 unique normal readings | 22,541 unique available readings, including 5,015 anomalous targets |
| Initial reference | No recording-specific initialization | First 60 readings establish a reference and receive no assessment |
| Alert rule | Three consecutive scores above the selected threshold | Five consecutive scores above the selected threshold |

Earlier exploration found that some normal validation readings differed from the normal operating conditions represented in training. Relative features were introduced to measure changes from each recording's starting condition. Gradient boosting then learns to distinguish labeled patterns in those features. It was selected because it achieved the highest precision among the candidates meeting the minimum recall requirements. Both the algorithm and the information available during training changed, along with the alert rule, so this comparison cannot attribute the improvement to gradient boosting alone.

These are development validation results. The same recordings were used to compare candidates and select thresholds, so the figures do not establish performance on new equipment or rule out overfitting to validation. SKAB annotations identify anomalous readings, not confirmed physical failures.

This notebook displays the saved aggregate results. Running it does not retrain a model, access test recordings or change the active detector. The [companion report](../docs/improvement-results.md) records the full comparison and reproduction commands.

In [ ]:
import json

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from forge.config import PROJECT_ROOT

result = json.loads((PROJECT_ROOT / "docs/improvement-results.json").read_text())
baseline = result["baseline_validation"]
selected = result["selected"]["metrics"]
pd.DataFrame({"Original Isolation Forest": baseline, "Relative gradient boosting": selected}).loc[
    [
        "precision",
        "recall",
        "f1",
        "fp",
        "fn",
        "false_alarm_onsets",
        "detected_events",
        "unavailable_rows",
    ]
]

## Reading the confusion matrices

Each matrix compares source annotations with persistent model alerts. Rows show whether SKAB labels a reading normal or anomalous. Columns show whether the detector raises an alert. The selected model reduces false-positive readings from 2,193 to 118, but missed anomalous readings increase from 351 to 1,198.

The new model makes no assessment during initialization. Its 519 normal initialization readings appear in the full matrix as normal readings without an alert. To avoid treating this unavailable period as successful detection, the report also calculates the false-positive rate (FPR) over available normal readings. Any anomalous initialization readings would still count as missed anomalies in recall.

In [ ]:
fig = make_subplots(
    rows=1, cols=2, subplot_titles=["Original Isolation Forest", "Relative gradient boosting"]
)
for column, metrics in enumerate([baseline, selected], start=1):
    matrix = [[metrics["tn"], metrics["fp"]], [metrics["fn"], metrics["tp"]]]
    fig.add_trace(
        go.Heatmap(
            z=matrix,
            x=["No alert", "Alert"],
            y=["Normal", "Anomaly"],
            text=matrix,
            texttemplate="%{text}",
            colorscale="Blues",
            showscale=False,
        ),
        row=1,
        col=column,
    )
fig.update_yaxes(autorange="reversed", title_text="Source annotation")
fig.update_layout(height=370, title="Development validation: 9,973 unique observations")
fig.show()

## How candidates were selected

The comparison required at least 60% recall across all anomalous readings, detection of at least 60% of annotated events, and at least 20% point recall in every validation group containing anomalies. Recordings that share observations belong to the same group. These requirements prevent a detector from achieving high precision simply by remaining almost silent.

Among candidates meeting those requirements, selection prioritizes precision, then fewer false alert onsets per normal hour, then recall. The plot shows each qualifying candidate at its selected threshold and persistence setting. Candidates without a qualifying setting remain in the table with `feasible=False`.

Model fitting uses training recordings only. Validation annotations select the candidate, threshold and number of consecutive above-threshold readings needed for an alert. The follow-up comparison was informed by the first stage, so this is an iterative development comparison.

In [ ]:
rows = []
for candidate in result["candidates"]:
    best = candidate["best"]
    rows.append(
        {
            "candidate": candidate["name"],
            "feasible": best is not None,
            "precision": best["metrics"]["precision"] if best else None,
            "recall": best["metrics"]["recall"] if best else None,
        }
    )
comparison = pd.DataFrame(rows)
fig = go.Figure(
    go.Scatter(
        x=comparison.recall,
        y=comparison.precision,
        text=comparison.candidate,
        mode="markers",
        marker={"size": 10},
    )
)
fig.add_hline(y=0.95, line_dash="dash", annotation_text="Precision target")
fig.update_layout(height=420, xaxis_title="Point recall", yaxis_title="Point precision")
fig.update_xaxes(range=[0, 1])
fig.update_yaxes(range=[0, 1])
fig.show()
comparison

## What the improvement leaves unresolved

The selected model reaches 95.1% precision at 65.8% point recall. It produces at least one new alert onset inside each of the ten annotated events, but it does not recover every anomalous reading within those events. The other/10 overlap group has only 21.7% point recall. Among detected events, the median delay before an alert is 41.5 seconds, compared with five seconds for the baseline. The two medians cover different sets of detected events.

There are still 26 false alert onsets, equivalent to 15.74 per available normal hour. The 95% precision target is met, but the 1% FPR and two-onsets-per-hour targets are not met. The table below shows where the remaining errors occur.

Initialization is another limitation. The first 60 readings of each source recording establish an unlabeled operating reference. After duplicate observations are removed, 519 validation readings are unavailable for assessment. None is annotated anomalous in this validation set. On another recording, a fault already present during initialization could become part of the reference and be missed.

In [ ]:
pd.DataFrame.from_dict(result["selected"]["full_validation_metrics"]["per_group"], orient="index")[
    [
        "tp",
        "fp",
        "fn",
        "tn",
        "precision",
        "recall",
        "detected_events",
        "events",
        "false_alarm_onsets",
    ]
]

## Reproduction and the next evaluation

After setting up the project and training the baseline, `python -m forge.ml.development` runs the first comparison and `python -m forge.ml.development --followup` runs the follow-up. Both write new local result folders without opening the test partition or activating a model. To use a selected result in the app, `python -m forge.ml.activation <run_id>` verifies its local artifact and updates the active model pointer. The original baseline is preserved.

Matching results on a rerun establishes reproducibility. It does not establish that the model generalizes. Because candidate and threshold selection used these validation recordings, the reported precision may be optimistic for unseen data. Software tests check specific implementation errors, including use of future readings, incorrect recording boundaries and labels entering the input features. They do not rule out statistical overfitting.

The next evaluation needs separately collected recordings, preferably from another operating period or asset. Freeze the model and alert settings before scoring them, then examine precision, recall, false alert frequency and delay together. The previously inspected test partition is not a fresh holdout for this work.